# Credit Modelling / Understanding Bank Soundness

# 1. NOVEL DATASET COLLECTION

## 1.1.1: Scraping IMF Policy Interest Rates

In [ ]:
#pip install pdftables.six
import pdftables
import numpy as np
import pandas as pd
from tabulate import tabulate
# Get page no. 1 from the pdf
pg1 = pdftables.get_pdf_page(open("Interest_Rates_IMF.pdf", "rb"),1)
# Get table from the pdf
table1 = pdftables.page_to_tables(pg1)
# Indexing the table
table1[0][2]
titles= zip(table1[0][0], table1[0][1]) #defining the titles
titles = list(titles)
# Convert the selected rows into pandas df
idf = pd.DataFrame([table1[0][8],table1[0][12],table1[0][16],table1[0][28],table1[0][31],table1[0][34],table1[0][40]])
#Scrapping the page no. 3 for rest of the years data.
pg3 = pdftables.get_pdf_page(open("Interest_Rates_IMF.pdf", "rb"),3)
# Scrapping the rest half of the table from page 3
table1_2 = pdftables.page_to_tables(pg3)
# Converting rows into pd df
idf2 = pd.DataFrame([table1_2[0][7],table1_2[0][11],table1_2[0][15],table1_2[0][27],table1_2[0][30],table1_2[0][33],table1_2[0][39]])
# Concate the dataframes to get whole data in one dataframe
idf3 = pd.concat([idf,idf2], axis=1)
# Providing columns name
idf3.columns = ["Country", "Scale", "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019","2020","2021","2022"]
# Scrapping the page no. 2 for rest of the countries data.
pg2 = pdftables.get_pdf_page(open("Interest_Rates_IMF.pdf", "rb"),2)
# Scrapping the rest half of the table from page 3
table2 = pdftables.page_to_tables(pg2)
# Converting selected rows into dataframe
idf4 = pd.DataFrame([table2[0][24],table2[0][42],table2[0][43]])
# page no. 4 for rest of the countries data.
pg4 = pdftables.get_pdf_page(open("Interest_Rates_IMF.pdf", "rb"),4)
# Scrapping the rest half of the table from page 4
table2_2 = pdftables.page_to_tables(pg4)
# Converting selected rows into dataframe
idf5 = pd.DataFrame([table2_2[0][24],table2_2[0][42],table2_2[0][43]])
# Concating the rest of the years data 
idf6 = pd.concat([idf4,idf5], axis = 1)
# Providing columns name
idf6.columns = ["Country", "Scale", "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019","2020","2021","2022"]
# Merging all the countries data together in a single dataframe
full_idf = pd.concat([idf3,idf6], ignore_index= True)
# Cleaning all '...' and '_' with np.NaN
full_idf.loc[[3,4], '2012']= np.NaN
full_idf.loc[[3,4], '2013']= np.NaN
full_idf.loc[3, '2014']= np.NaN
full_idf.loc[[3,4], '2015']= np.NaN
full_idf.loc[[2,3,4], '2016']= np.NaN
full_idf.loc[[2,3,4,9], '2017']= np.NaN
full_idf.loc[[2,3,4,9], '2018']= np.NaN
full_idf.loc[[2,3,4,9], '2019']= np.NaN
full_idf.loc[[2,3,4,9], '2020']= np.NaN
full_idf.loc[[2,4,5,9], '2021']= np.NaN
full_idf.loc[[4,6,7,9], '2022']= np.NaN

# Shortening the country names
full_idf.loc[0, 'Country']= "Azerbaijan"
full_idf.loc[1, 'Country']= "Belarus"
full_idf.loc[7, 'Country']= "Poland"
full_idf.loc[0, 'Country']= "Azerbaijan"
full_idf.loc[8, 'Country']= "Turkiye"
full_idf.loc[9, 'Country']= "UK"
# Dropping column 'Scale'
full_idf = full_idf.drop(columns = "Scale", axis = 1)
full_idf
# Convert to a Latex table
table = tabulate(full_idf,headers = full_idf.columns, tablefmt = "latex")
print(table)
full_idf

## 1.1.2: Scraping BICRA

In [ ]:
import requests
from bs4 import BeautifulSoup
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()
import os
username = os.getenv("SPGLOBAL_USERNAME")
password = os.getenv("SPGLOBAL_PASSWORD")

login_url = os.getenv("SPGLOBAL_LOGIN_URL")
data_url_2021 = "https://www.spglobal.com/ratings/en/research/articles/220128-banking-industry-country-risk-assessment-update-january-2022-12258313"

login_data = {
    "username": username,
    "password": password
}

# Log in to the website and fetch the page
session = requests.Session()
response = session.post(login_url, data=login_data)
response = session.get(data_url_2021)

# Parse
soup = BeautifulSoup(response.text, "html.parser")
table = soup.find("table", class_="article")
rows = table.find_all("tr")

# Extract the table data
table_data = []
for row in rows:
    cells = row.find_all(["td", "th"])  # Find all table cells and header cells
    cells_text = [cell.get_text(strip=True) for cell in cells]
    table_data.append(cells_text)

for row in table_data:
    print("\t".join(row))

# Df
import pandas as pd
df = pd.DataFrame(table_data)
# 1st row as the column names and remove it from the data
df.columns = df.iloc[0]
df = df.drop(0)
df = df.reset_index(drop=True)
df

# Section 2: DATA CLEANING, CHECKING AND ORGANIZATION

## 2.1. Management Ratios Calculation and Data Compilation

### 2.1.1. Splitting Management Data

In [ ]:
import pandas as pd
# Load the Excel file into a pandas dataframe
df = pd.read_excel('Data_Management_2.xlsx', sheet_name='Sheet1')
# Drop the first two rows as there were empty rows
df = df.drop(index=[0])
# Reset the index
df = df.reset_index(drop=True)
# Create a new dataframe with splitting the initial 3 columns
df1 = df.iloc[:, :3]
# Create a new dataframe with rest of the columns
df2 = df.iloc[:, 3:]
# Assign Row 0 as a Headline
df1.columns = df1.iloc[0]


### 2.1.2. Transposed to Manipulate Data and Classify by Years

In [ ]:
# 1.2 Transposed to Manipulate Data and Classify by Years

# Transposed it to classify by years
df_t = df2.transpose()
# Deleted FY from the Year column index 1
df_t[1] = df_t[1].str.replace("FY", "")
# Sorted Second Column for Reverse Order from 2021 to 2010
df_t = df_t.sort_values(by=1, ascending=False) # make it false to create descending
# Replace specified values in all rows of column 0
replacement_dict = {273925: 'OPERATING INCOME',
                    '273925': 'OPERATING INCOME',
                    '279005': 'OPERATING EXPENSE',
                    '278992': 'INTEREST EXPENSE',
                    '278788': 'LOAN LOSS RESERVE',
                    'SP_TOTAL_ASSETS': 'TOTAL ASSETS'}

def replace_values(val):
    if val in replacement_dict:
        return replacement_dict[val]
    return val
df_t[0] = df_t[0].apply(replace_values)

# ReTranspose the modified version
df_tt = df_t.transpose()
# Assign Row 0 as a Headline
df_tt.columns = df_tt.iloc[0]

### 2.1.3. Splitting Data by Years

In [ ]:
# 1.3. SPLITTING DATA BY YEARS

# This is a loop for splitting a data for each years. 
dfs = {}

for i, year in enumerate(range(2021, 2009, -1)):
    start_col = i * 5
    end_col = start_col + 5
    
    temp_df = df_tt.iloc[:, start_col:end_col] #we sliced every 5 of it to split for years
    temp_df = pd.concat([df1, temp_df], axis=1) #merge to add entity name country name and id
    temp_df.insert(loc=2, column='YEAR', value=year) #added related years
    temp_df = temp_df.iloc[2:]
    
    dfs[year] = temp_df

# Then this part puts related year df
man21 = dfs[2021]
man20 = dfs[2020]
man19 = dfs[2019]
man18 = dfs[2018]
man17 = dfs[2017]
man16 = dfs[2016]
man15 = dfs[2015]
man14 = dfs[2014]
man13 = dfs[2013]
man12 = dfs[2012]
man11 = dfs[2011]
man10 = dfs[2010]


### 2.1.4. Calculating Ratios

Following code generates the data frames dictionary, which contains all of the DataFrames. 
It then loops through the dictionary, performing the same operations on each DataFrame. 
Finally, it unpacks the updated DataFrames and assigns them back to variables. 
The reason we are doing this to reach out missing management ratios from using the related metrics.

In [ ]:
# 1.3.2 Calculating Ratios
management_frames = {
    'man21': man21,
    'man20': man20,
    'man19': man19,
    'man18': man18,
    'man17': man17,
    'man16': man16,
    'man15': man15,
    'man14': man14,
    'man13': man13,
    'man12': man12,
    'man11': man11,
    'man10': man10}

for key, df in management_frames.items():
    df = df[df['TOTAL ASSETS'] != 0]
    df['OPERATING RETURN ON ASSETS'] = df['OPERATING INCOME'] / df['TOTAL ASSETS']*100
    
    # Filter out rows where the denominator is zero
    df = df[df['OPERATING INCOME'] - df['LOAN LOSS RESERVE'] != 0]
    df['NON-OPERATING EXPENSE RATIO'] = (df['OPERATING EXPENSE'] - 
                                         df['INTEREST EXPENSE']) / (df['OPERATING INCOME'] - 
                                                                    df['LOAN LOSS RESERVE'])*100
    
    # Drop columns 4 to 8
    columns_to_drop = df.columns[4:9]
    df.drop(columns_to_drop, axis=1, inplace=True)
    
    management_frames[key] = df

man21, man20, man19, man18, man17, man16, man15, man14, man13, man12, man11, man10 = management_frames.values()


In [ ]:
man10.tail()

### 2.1.5. Merging Management Ratios with other CA(M)ELS Ratios

In [ ]:
# Load the CAMELS files
CAMELS_2021 = pd.read_excel('CAMELS_2021.xlsx')
CAMELS_2020 = pd.read_excel('CAMELS_2020.xlsx')
CAMELS_2019 = pd.read_excel('CAMELS_2019.xlsx')
CAMELS_2018 = pd.read_excel('CAMELS_2018.xlsx')
CAMELS_2017 = pd.read_excel('CAMELS_2017.xlsx')
CAMELS_2016 = pd.read_excel('CAMELS_2016.xlsx')
CAMELS_2015 = pd.read_excel('CAMELS_2015.xlsx')
CAMELS_2014 = pd.read_excel('CAMELS_2014.xlsx')
CAMELS_2013 = pd.read_excel('CAMELS_2013.xlsx')
CAMELS_2013 = pd.read_excel('CAMELS_2013.xlsx')
CAMELS_2013 = pd.read_excel('CAMELS_2013.xlsx')
CAMELS_2012 = pd.read_excel('CAMELS_2012.xlsx')
CAMELS_2011 = pd.read_excel('CAMELS_2011.xlsx')
CAMELS_2010 = pd.read_excel('CAMELS_2010.xlsx')


In [ ]:

import pandas as pd

years = ['2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010']
dfs = []

for year in years:
    filename = 'CAMELS_' + year + '.xlsx'
    df = pd.read_excel(filename)
    dfs.append(df)

# concatenate all the dataframes
CAMELS = pd.concat(dfs, ignore_index=True)


### 2.1.6. Data Compilation

In [ ]:
# 1.6 We created dictionary to merge easily which will be similar to Vlookup Function typically matches SP_ENTITY_ID in both df and merge accordingly
# Then drops
CAMELS_frames = {
    'CAMELS_2021': CAMELS_2021,
    'CAMELS_2020': CAMELS_2020,
    'CAMELS_2019': CAMELS_2019,
    'CAMELS_2018': CAMELS_2018,
    'CAMELS_2017': CAMELS_2017,
    'CAMELS_2016': CAMELS_2016,
    'CAMELS_2015': CAMELS_2015,
    'CAMELS_2014': CAMELS_2014,
    'CAMELS_2013': CAMELS_2013,
    'CAMELS_2012': CAMELS_2012,
    'CAMELS_2011': CAMELS_2011,
    'CAMELS_2010': CAMELS_2010
}

management_frames = {
    'man21': man21,
    'man20': man20,
    'man19': man19,
    'man18': man18,
    'man17': man17,
    'man16': man16,
    'man15': man15,
    'man14': man14,
    'man13': man13,
    'man12': man12,
    'man11': man11,
    'man10': man10
}



In [ ]:
merged_frames = {}

# Similar to Vlookup Function typically matches SP_ENTITY_ID in both CAMELS and and merge accordingly
for year, (camels_key, camels_df), (man_key, man_df) in zip(range(2021, 2009, -1), CAMELS_frames.items(), management_frames.items()):
    merged_df = pd.merge(camels_df, 
                         man_df, 
                         left_on='SP_ENTITY_ID', 
                         right_on='SP_ENTITY_ID', 
                         how='left')
    merged_df.drop(merged_df.columns[-5:-2], axis=1, inplace=True)
    merged_frames[f"merged_{year}"] = merged_df

merged_2021, merged_2020, merged_2019, merged_2018, merged_2017, merged_2016, merged_2015, merged_2014, merged_2013, merged_2012, merged_2011, merged_2010 = merged_frames.values()


### 2.1.7. Defining New Column Names

In [ ]:
# Define the new column names
new_columns = ["BANK NAME", 
               "YEAR",
               "ID",
               "TYPE",
               "COUNTRY",
               "CCODE",
               "GEOGRAPHY",
               "TOTAL CAPITAL RATIO",
               "TIER1 RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAA",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "LIQUIDITY COVERAGE RATIO",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO"]

# Rename the columns for each DataFrame in the merged_frames dictionary
for key, df in merged_frames.items():
    df.columns = new_columns


## 2.2. Cleaning, Checking and Organization

### 2.2.1. Checking for missing values

In [ ]:
#1.9 Checking Missing Values Graphical Example
import missingno as msno
msno.matrix(merged_2020)


In [ ]:

import missingno as msno
msno.bar(merged_2020)
missing_20 = merged_2020.isnull().sum()
print(missing_20)

### 2.2.2. Dropping Banks with All Ratios are Missing

In [ ]:
# 1.9 for Dropped rows containing NaN values in all columns between 8 and 20 (inclusive)
merged_frames = {}

# Vlookup Function typically matches SP_ENTITY_ID in both CAMELS and and merge accordingly
for year, (camels_key, camels_df), (man_key, man_df) in zip(range(2021, 2009, -1), CAMELS_frames.items(), management_frames.items()):
    merged_df = pd.merge(camels_df, 
                         man_df, 
                         left_on='SP_ENTITY_ID', 
                         right_on='SP_ENTITY_ID', 
                         how='left')
    merged_df.drop(merged_df.columns[-5:-2], axis=1, inplace=True)
      
    # Dropped rows containing NaN values in all columns between 8 and 20 (inclusive)
    merged_df.dropna(subset=merged_df.columns[7:21], how='all', inplace=True)
    
    merged_frames[f"merged_{year}"] = merged_df

merged_2021, merged_2020, merged_2019, merged_2018, merged_2017, merged_2016, merged_2015, merged_2014, merged_2013, merged_2012, merged_2011, merged_2010 = merged_frames.values()

# Define the new column names
new_columns = ["BANK NAME", 
               "YEAR",
               "ID",
               "TYPE",
               "COUNTRY",
               "CCODE",
               "GEOGRAPHY",
               "TOTAL CAPITAL RATIO",
               "TIER1 RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAA",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "LIQUIDITY COVERAGE RATIO",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO"]

# Rename the columns for each DataFrame in the merged_frames dictionary
for key, df in merged_frames.items():
    df.columns = new_columns

#### Re-check

In [ ]:
missing_20 = merged_2020.isnull().sum()
print(missing_20)

In [ ]:
import missingno as msno
msno.bar(merged_2020)
msno.matrix(merged_2020)

### 2.2.3. Dealing with Rest of Missing Values

Our Approach to when we examine the data set, it has been determined that the lost data are either concentrated in the country by country
and are caused by the lack of financial reporting quality (for example, Liquidity Coverage Ratio Azerbaijan), 
and belong to banks that are small in terms of scale or have poor performance in terms of asset size, 
on a yearly basis or in a certain country. 
Banks with missing data were grouped by country while filling in missing data, 
and the median of the 5 banks with the worst performance was taken in order to penalize banks with missing data. Since our data has right-skewness (positive skewness) we have chosen median to punish the banks with reporting issues.


In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

merged_frames = {
    2021: merged_2021,
    2020: merged_2020,
    2019: merged_2019,
    2018: merged_2018,
    2017: merged_2017,
    2016: merged_2016,
    2015: merged_2015,
    2014: merged_2014,
    2013: merged_2013,
    2012: merged_2012,
    2011: merged_2011,
    2010: merged_2010}
    
cols_to_fill = ["TOTAL CAPITAL RATIO",
               "TIER1 RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAA",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "LIQUIDITY COVERAGE RATIO",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO"]

# Iterate through years and apply the code snippet to each dataframe
for year in range(2010, 2022):
    df = merged_frames[year]
    for col in cols_to_fill:
        df[col] = df[col].replace('NM', float('nan'))
        df[col] = df[col].astype(float)
        df[col].fillna(df.groupby('COUNTRY')[col].nsmallest(5).median(), inplace=True)
        imputer = IterativeImputer()
        df[col] = imputer.fit_transform(df[[col]])
   
    df = df[df['TOTAL CAPITAL RATIO'] <= 100]
    merged_frames[year] = df

df

In [ ]:
# Convert "TOTAL CAPITAL RATIO" column to numeric values
merged_2021["TIER1 RATIO"] = pd.to_numeric(merged_2021["TIER1 RATIO"], errors='coerce')
# Drop rows with NaN values
merged_2021.dropna(subset=["TIER1 RATIO"], inplace=True)
#Find the 5 lowest "TOTAL CAPITAL RATIO" values for each country and calculate the average
#result = merged_2021.groupby("COUNTRY")["TIER1 RATIO"].apply(lambda x: x.nsmallest(5).mean())
result = merged_2021.groupby("COUNTRY")["TIER1 RATIO"].apply(lambda x: x.nsmallest(5).mean())

# Print the result
print(result)

### 2.2.4. Merging All Data and Adding BICRA

In [ ]:
# MERGING ALL DATA AND ADDING BICRA COLUMNS
import pandas as pd

# Iterate through years from 2021 to 2010 (in reverse order) and save each dataframe as a CSV file
for year in range(2021, 2009, -1):
    # Get the corresponding dataframe from the merged_frames dictionary
    merged_df = merged_frames[year]
    # Save the dataframe as a CSV file with the name merged_{year}.csv
    merged_df.to_csv(f"merged_{year}.csv", index=False)

# Create an empty dictionary to store the dataframes
dfs = {}

# Load the CSV files into dataframes and store them in the dictionary
for year in range(2010, 2022):
    df = pd.read_csv(f"merged_{year}.csv")
    dfs[year] = df

# Concatenate the dataframes into a single dataframe
merged_df = pd.concat(dfs.values())

# Import BICRA file
BICRA_df = pd.read_excel('BICRA.xlsx')

# Temp only for index (COUNTRY, YEAR) and BICRA as its column
temp_df = BICRA_df.set_index(['COUNTRY', 'YEAR'])['BICRA']
merged_df['BICRA'] = merged_df.set_index(['COUNTRY', 'YEAR']).index.map(temp_df)

# Save the merged dataframe as an Excel file
merged_df.to_excel("merged_data.xlsx", index=False)

msno.bar(merged_df)

In [ ]:
msno.bar(merged_df)

# 3. DATABASE CREATION AND QUERYING

In [ ]:
import pandas as pd
import sqlalchemy #sqltoolkit 
import sqlite3 as lite
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Establishing the connection with the database file
con = lite.connect('CAMELS.db', timeout=10)
cur = con.cursor()
# Reading the bank data from the excel file to pandas dataframe
bankdf = pd.read_excel("Banks.xlsx")
# Fixing the column names
bankdf.rename(columns = {
    "SP ENTITY ID": 'id',
    "ENTITY NAME": "Bank_Name",
    "INDUSTRY CLASSIFICATION":"Industry_Class",
    "CCODE":"country_code"
}, inplace = True)
# Transforming the DataFrame into a database table
bankdf.to_sql("Banks", con, index = False, if_exists = "replace")
# Reading the country data from the excel file to pandas dataframe
countrydf = pd.read_excel("Country.xlsx")
# Fixing the column names
countrydf.rename(columns = {
  "COUNTRY": "Country",
    "GEOGRAPHY":"Geography",
    "CCODE":"c_code"
}, inplace = True)

# Cleaning the duplicate country rows
countrydf.drop_duplicates(subset= ["Country","c_code", "Geography"], inplace = True)
# Transforming the country data into a database table
countrydf.to_sql("Countries", con, index = False, if_exists = "replace")
# Trying to query the table from database
df = pd.read_sql_query("select * from Banks", con)
# Reading the Ratio data from the excel file to pandas dataframe
ratiodf = pd.read_excel("Ratios.xlsx")
# Creating the Ratios table in the database
ratiodf.to_sql("Ratios", con, index = False, if_exists = "replace")
# Tried to run a general query to the database
df = pd.read_sql_query("""SELECT Ratios.YEAR, Ratios.TCR,Banks.Bank_Name FROM Ratios 
                       JOIN Banks ON Ratios.ENTITY_ID = Banks.id GROUP BY YEAR, Bank_Name""", con)
# Querying the whole data into one table from the whole database
query = """SELECT 
    Banks.Bank_Name,
    Countries.Country, 
    Countries.c_code,
    Ratios.YEAR,
    Ratios.TCR,
    Ratios.TIER1_RATIO,
    Ratios.TEXAS_RATIO, 
    Ratios.PROBLEM_LOANS_RATIO,
    Ratios.LL_RESERVE_TO_PL,
    Ratios.NET_INTEREST_MARGIN,
    Ratios.ROAA,
    Ratios.ROAE,
    Ratios.PL_to_TANGIBLE_EQUITYnRESERVES,
    Ratios.debt_to_equity,
    Ratios.LIQUIDITY_COVERAGE_RATIO, 
    Ratios.BOOKVALUE_PER_SHARE_GROWTH,
    Ratios.OPERATING_RETURN_ON_ASSETS,
    Ratios.NON_OPERATING_EXPENSE_RATIO,
    Ratios.BICRA
FROM Ratios
JOIN Banks ON Ratios.ENTITY_ID = Banks.id 
JOIN Countries ON Banks.country_code = Countries.c_code 
GROUP BY Ratios.YEAR, Banks.Bank_Name;
"""

df = pd.read_sql_query(query, con)
# Creating an interecting report of the whole dataset using Dataprep
import dataprep.eda
from dataprep.eda import create_report
create_report(df).show()
# Calculation of correlatin matrix of the data queried
corr_matrix = df.corr()
# Set up the size and style of the heatmap
plt.figure(figsize=(16, 12))
sns.set(style="white")
# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="coolwarm",
    center=0.1,
    square=True,
    linewidths=0.1,
    annot=True,  # Show the correlation values
    fmt=".2f",  # Format the correlation values
    cbar_kws={"shrink": 0.5},
)
plt.title("Correlation Heatmap", fontsize=14, fontweight='bold')
plt.show()
# Identifying groups of variables that are highly correlated with each other.
sns.clustermap(corr_matrix)

# 4. Data Visualization

### 4.1. Average of 5 Lowest Tier1 Ratio

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="dark", font_scale=1.2)

for year in range(2019, 2022):
    # Convert "TIER1 RATIO" column to numeric values and drop rows with NaN values
    merged_frames[year]["TIER1 RATIO"] = pd.to_numeric(merged_frames[year]["TIER1 RATIO"], errors='coerce')
    merged_frames[year].dropna(subset=["TIER1 RATIO"], inplace=True)
    
    # Find the 5 lowest "TIER1 RATIO" values for each country and calculate the average
    result = merged_frames[year].groupby("COUNTRY")["TIER1 RATIO"].apply(lambda x: x.nsmallest(5).mean())

    # Create a bar chart of the result for each year
    plt.figure(figsize=(12, 6))
    sns.barplot(x=result.index, y=result.values, palette="coolwarm")
    plt.title(f'Average of 5 Lowest TIER1 RATIOS by Country in {year}', fontweight='bold')
    plt.xlabel('Country', fontweight='bold')
    plt.ylabel('Average TIER1 RATIO', fontweight='bold')
    plt.xticks(rotation=45, fontsize=10, fontweight='normal')
    plt.yticks(fontsize=10, fontweight='bold')
    plt.tight_layout()
    
    # Display the chart
    plt.show()


### 4.4.2. Average Problem Loans by Country

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(14, 8))

for year in range(2010, 2022):
    # Turn "PROBLEM LOANS RATIO" to numeric values and drop NaN 
    merged_frames[year]["PROBLEM LOANS RATIO"] = pd.to_numeric(merged_frames[year]["PROBLEM LOANS RATIO"], errors='coerce')
    merged_frames[year].dropna(subset=["PROBLEM LOANS RATIO"], inplace=True)
    # Average Calcuation
    avg_problem_loans = merged_frames[year].groupby("COUNTRY")["PROBLEM LOANS RATIO"].mean()
    # Create a scatter plot for each year
    plt.scatter(avg_problem_loans.index, avg_problem_loans.values, label=f'{year}')
# Set the title, labels, and legend
plt.title('Average PROBLEM LOANS RATIO by Country (2010-2021)')
plt.xlabel('Country')
plt.ylabel('Average PROBLEM LOANS RATIO')
plt.xticks(rotation=45)
plt.legend(title='Year')
# Display the chart
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define a list of colors
colors = plt.cm.Reds(np.linspace(0.2, 1, 12))

plt.figure(figsize=(14, 8))

for i, year in enumerate(range(2010, 2022)):
    # Convert "PROBLEM LOANS RATIO" column to numeric drop rows with NaN values
    merged_frames[year]["PROBLEM LOANS RATIO"] = pd.to_numeric(merged_frames[year]["PROBLEM LOANS RATIO"], errors='coerce')
    merged_frames[year].dropna(subset=["PROBLEM LOANS RATIO"], inplace=True)
    # Calculate the average "PROBLEM LOANS RATIO" for each country
    avg_problem_loans = merged_frames[year].groupby("COUNTRY")["PROBLEM LOANS RATIO"].mean()
    plt.scatter(avg_problem_loans.index, avg_problem_loans.values, label=f'{year}', c=colors[i])
# title, labels, and legend
plt.title('Average PROBLEM LOANS RATIO by Country (2010-2021)')
plt.xlabel('Country')
plt.ylabel('Average PROBLEM LOANS RATIO')
plt.xticks(rotation=45)
plt.legend(title='Year')

# Display the chart
plt.show()


### 4.3. Correlation Maps

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

corr_matrix = merged_df.corr()
# Set up the size and style of the heatmap
plt.figure(figsize=(16, 12))
sns.set(style="white")
# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
# Triu Returns a copy of an array with the elements below the k-th diagonal zeroed. 
# For arrays with ndim exceeding 2, triu will apply to the final two axes. This code is taken from Github
# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="coolwarm",
    center=0.0,
    square=True,
    linewidths=0.1,
    annot=True,  # Show the correlation values
    fmt=".2f",  # Format the correlation values
    cbar_kws={"shrink": 0.5},
)
plt.title("Correlation Heatmap", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
df = pd.read_sql_query(query, con)
# Creating an interecting report of the whole dataset using Dataprep
import dataprep.eda
from dataprep.eda import create_report
create_report(df).show()
# Calculation of correlatin matrix of the data queried
corr_matrix = df.corr()
# Set up the size and style of the heatmap
plt.figure(figsize=(16, 12))
sns.set(style="white")
# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
corr_matrix,
mask=mask,
cmap="coolwarm",
center=0.1,
square=True,
linewidths=0.1,
annot=True, # Show the correlation values
fmt=".2f", # Format the correlation values
cbar_kws={"shrink": 0.5},
)
plt.title("Correlation Heatmap", fontsize=14, fontweight='bold')
plt.show()
# Identifying groups of variables that are highly correlated with each other.
sns.clustermap(corr_matrix)

# 5. TEXTUAL ANALYSIS

In [ ]:
pip install newsapi-python

In [ ]:
import nltk
#nltk.download('vader_lexicon')
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd
from newsapi import NewsApiClient
#from newsapi.newsapi_client import NewsApiClient
from datetime import date, timedelta, datetime

from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
pd.set_option('display.max_colwidth',1000)

NEWS_API_KEY = os.getenv("NEWS_API_KEY")
#https://newsapi.org/docs/endpoints/everything
newsapi = NewsApiClient(api_key=os.getenv("NEWS_API_KEY"))
keywords = ['Performance of European Banks']
my_date = date.today() - timedelta(days=30) 
for keyword in keywords:
      articles = newsapi.get_everything(q = keyword,
                                      from_param = my_date.isoformat(), 
                                      to = (my_date + timedelta(days = 30)).isoformat(),
                                      language="en",
                                      #sources = ",".join(sources_list),
                                      sort_by="relevancy",
                                      page_size = 100)
articles

In [ ]:
import matplotlib.pyplot as plt
# Creating funtion to get sentiment from the articles
def get_articles_sentiments(keyword, startd, sources_list = None, show_all_articles = False):
newsapi = NewsApiClient(api_key=os.getenv("NEWS_API_KEY"))
  if type(startd) == str:
    my_date = datetime.strptime(startd,'%d-%b-%Y')
  else:
    my_date = startd
  # business_en_sources = get_sources('business','en')
  if sources_list:
    articles = newsapi.get_everything(q = keyword,
                                      from_param = my_date.isoformat(), 
                                      to = (my_date + timedelta(days = 30)).isoformat(),
                                      language="en",
                                      sources = ",".join(sources_list),
                                      sort_by="relevancy",
                                      page_size = 100)
  else:
     articles = newsapi.get_everything(q = keyword,
                                       from_param = my_date.isoformat(), 
                                       to = (my_date + timedelta(days = 30)).isoformat(),
                                       language="en",
                                       sort_by="relevancy",
                                       page_size = 100)
  article_content = ''

  date_sentiments = {}
  date_sentiments_list = []
  seen = set()
  
  for article in articles['articles']:
    if str(article['title']) in seen:
      continue
    else:
      seen.add(str(article['title']))
      article_content = str(article['title']) + '. ' + str(article['description'])      
      sentiment = sia.polarity_scores(article_content)['compound']
      date_sentiments.setdefault(my_date, []).append(sentiment)
      date_sentiments_list.append((sentiment, article['url'],article['title'],article['description']))

  date_sentiments_l = sorted(date_sentiments_list, key=lambda tup: tup[0], reverse = True)   
  sent_list = list(date_sentiments.values())[0]
  return pd.DataFrame(date_sentiments_list, columns=['Sentiment','URL','Title','Description'])
return_articles = get_articles_sentiments(keyword= 'Performance of European Banks' ,startd = '21-Mar-2023', 
sources_list = None, show_all_articles= True)
# Note!
#Last metric we used 16 Mar 2023. Please change it if you want use newer date!

print(return_articles.Sentiment.mean()) # Returns the mean sentiment
print(return_articles.Sentiment.count()) # Returns the count of the sentiment
print(return_articles.Description) # Returns the description

# Get sources with english Title, Description
my_date = date.today() - timedelta(days=30) 
return_articles = get_articles_sentiments(keyword= 'Performance of European Banks' ,startd = my_date, sources_list = None, 
show_all_articles= True)
print(return_articles)
# print(return_articles.Sentiment.mean())
# print(return_articles.Sentiment.count())
# print(return_articles.Description)
# Histogram plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(return_articles.Sentiment, bins=30, color='skyblue', alpha=0.8, histtype='stepfilled')
# Axis labels and a title
ax.set_xlabel('Sentiment')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Sentiment Values')
# Display the plot
plt.show()
return_articles.head()
#Counting the total number of positives, negatives, and neutral of sentiments
pos_sentiment = 0
neg_sentiment = 0
neutral = 0
for row in return_articles["Sentiment"]:
  if row > 0:
    pos_sentiment += 1
  elif row < 0:
    neg_sentiment += 1
  else:
    neutral += 1
print(f"Positive Sentiment: {pos_sentiment}")
print(f"Negative Sentiment: {neg_sentiment}")
print(f"Neutral Sentiment: {neutral}")

# Creating a PIE Chart for visualisation
labels = ['Positive', 'Negative', 'Neutral']
values = [pos_sentiment, neg_sentiment, neutral]
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(values, labels=labels, autopct='%1.1f%%', startangle=90)
ax.axis('equal')
ax.set_title('Proportions of sentiments')

WARNING: Please note you can only get last one months news otherwise code will not work

# 6. MODELING USING MACHINE LEARNING

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import tree
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, precision_score, roc_curve, roc_auc_score, auc, mean_squared_error
import graphviz

import warnings
warnings.filterwarnings('ignore')

## 6.1. Logistic Regression

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Load the dataset
default = pd.read_excel('merged_data.xlsx')
# Add a new column to the DataFrame based on TOTAL CAPITAL RATIO
default["RISK_12"] = np.where(default["TOTAL CAPITAL RATIO"] >= 12, 0, 1) #If greater than 12 less risky
default = pd.get_dummies(default, columns = ["TYPE"], prefix = ["TYPE"])
# Define the independent and dependent variables
X = default[["TIER1 RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAA",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "LIQUIDITY COVERAGE RATIO",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO",
               "BICRA",
               "TYPE_Diversified Commercial Banks"
               ]]
y = default.RISK_12
# Create a logistic regression model using sklearn
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression()
# Fit the logistic regression model using statsmodels on the training set (This is for statistic results)
X_train_sm = sm.add_constant(X_train)
logit_model = sm.Logit(y_train, X_train_sm)
result = logit_model.fit()
# Fit the logistic regression model using sklearn
model.fit(X_train, y_train)
# k-fold cross-validation
k = 5
scores = cross_val_score(model, X, y, cv=k, scoring='roc_auc')
# Print accuracy scores
print(f"Accuracy scores for {k}-fold cross-validation: {scores}")
#print(f"Mean accuracy: {scores.mean()}")
# Predictions on the entire dataset
y_pred = model.predict(X)
# print(classification_report(y, y_pred))
# Predicted probabilities
y_prob = model.predict_proba(X)[:, 1]
# false positive rate, true positive rate, and thresholds
fpr, tpr, thresholds = roc_curve(y, y_prob)
# Compute the area under the ROC curve
roc_auc = roc_auc_score(y, y_prob)
print(f"Area under the ROC curve: {roc_auc}")
# Plot the ROC curve
plt.plot(fpr, tpr, label='AUC = %0.3f' % roc_auc)
plt.plot([0, 1], [0, 1], 'k--', color='r', label="Random Classifier")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression ROC curve')
plt.legend(loc="lower right")
plt.show()
print(classification_report(y, y_pred, target_names=['No', 'Yes']))
# Print the logistic regression results from statsmodels
print(result.summary())


## 6.2. Classification Tree

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import tree
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
import graphviz
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, precision_score, roc_curve, auc, mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV

import warnings
warnings.filterwarnings('ignore')

### 6.2.1. Finding Maximum Leaf Nodes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import tree
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
from sklearn.metrics import confusion_matrix, classification_report, precision_score, roc_curve, auc, mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV
import warnings
warnings.filterwarnings('ignore')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
regr = tree.DecisionTreeRegressor()
regr.fit(X_train, y_train)

param_grid = {'max_leaf_nodes': [2, 3, 4, 5, 6, 7, 8, 9, 10]}

# Grid search with cross-validation for max lead nodes
grid_search = GridSearchCV(estimator=regr, param_grid=param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X, y)

print("Best hyperparameters:", grid_search.best_params_)


## 6.2.2. Fitting and Plotting Decision Tree

In [ ]:
# Fit the decision tre
regr = tree.DecisionTreeClassifier(max_leaf_nodes=9)
regr.fit(X, y)
# Plot the decision tree
plt.figure(figsize=(15,10))
tree.plot_tree(regr,fontsize=10,feature_names=X.columns,filled=True,
               rounded = True,precision=2)
plt.show()

## 6.2.3. Model Accuracy

In [ ]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
# Make predictions on the test set
y_pred = regr.predict(X_test)
# Generate the classification report
# cr = classification_report(y_test, y_pred)
# print("Classification report:\n", cr)
# Generate the ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)
print("ROC AUC:", roc_auc)

## 6.3. Random Forest

### 6.3.1. Finding Optimal Hyperparameter

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {'max_features': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}

# Define the random forest classifier
regr_RF = RandomForestClassifier(random_state=42)

# Define the grid search object with cross-validation
grid_search_RF = GridSearchCV(estimator=regr_RF, param_grid=param_grid, cv=5, scoring='roc_auc')

# Fit the grid search object to the training data
grid_search_RF.fit(X_train, y_train)

# Print the best hyperparameters and the corresponding score
print("Best hyperparameters:", grid_search_RF.best_params_)


### 6.3.1. Finding ROC

In [ ]:
from sklearn.metrics import roc_auc_score
# Train the random forest classifier with the best max_features
regr_RF = RandomForestClassifier(max_features=6, random_state=42)
regr_RF.fit(X_train, y_train)

# Predict the probabilities for the test set
y_test_probs_RF = regr_RF.predict_proba(X_test)[:, 1]
# Calculate the ROC AUC score
roc_auc_RF = roc_auc_score(y_test, y_test_probs_RF)
print("ROC AUC score for random forest classifier:", roc_auc_RF)

## 6.4. XGBOOST

In [ ]:
### XGBOOST
# pip install xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import plot_importance
import matplotlib.pyplot as plt

# Define the independent and dependent variables
X = default[["TIER1 RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAA",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "LIQUIDITY COVERAGE RATIO",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO",
               "BICRA",
               "TYPE_Diversified Commercial Banks"
               ]]
y = default.RISK_12

# Split the dataset into training and test datasets
train, test = train_test_split(default, test_size=0.3, shuffle=False)

# Define the feature and target variables for the training and test datasets
X_train = train.loc[:, X.columns]
y_train = train.loc[:, 'RISK_12']
X_test = test.loc[:, X.columns]
y_test = test.loc[:, 'RISK_12']

# Train the XGBoost model on the training dataset
XGB = XGBClassifier(n_estimators=50, n_jobs=-1, verbose=1)
XGB.fit(X_train, y_train, eval_metric=["auc"], eval_set=[(X_train, y_train), (X_test, y_test)])

# Plot feature importance
plot_importance(XGB)
plt.show()

# Compute and print the area under the receiver operating characteristic (ROC) curve for the training and test datasets
y_train_pred = XGB.predict_proba(X_train)[:, 1]
y_test_pred = XGB.predict_proba(X_test)[:, 1]
print('Training ROC:' + "{:.3f}".format(roc_auc_score(y_train, y_train_pred)))
print('Test ROC:' + "{:.3f}".format(roc_auc_score(y_test, y_test_pred)))


In [ ]:
%pip install dataprep
from dataprep.eda import create_report

## 6.5. Unsupervised Model: K-Means with PCA

 SelectKBest method with mutual information as the scoring function. Mutual information measures the dependency between features and can be useful for feature selection in unsupervised learning

### 6.5.1. Finding Optimum Clusters

In [ ]:
# Load the dataset
default = pd.read_excel('merged_data.xlsx')
default = pd.get_dummies(default, columns = ["TYPE"], prefix = ["TYPE"])


In [ ]:
#pip install yellowbrick
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer

# Define the feature columns
X_cols = ["TOTAL CAPITAL RATIO",
               "TIER1 RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAA",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "LIQUIDITY COVERAGE RATIO",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO",
               "BICRA",
               "TYPE_Diversified Commercial Banks"]

# Extract the features
X = default[X_cols]

kmeans = KMeans()
visualizer = KElbowVisualizer(kmeans, k=(3,11))
visualizer.fit(X) 
visualizer.poof()  

### 6.5.2. K-Means with Principal Component Analysis (PCA)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Define the feature columns
X_cols = ["TOTAL CAPITAL RATIO",
               "TIER1 RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAA",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "LIQUIDITY COVERAGE RATIO",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO",
               "BICRA",
               "TYPE_Diversified Commercial Banks"]

# Extract the features
X = default[X_cols]

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA for dimensionality reduction
pca = PCA(n_components=0.95)  # Keep 95% of the explained variance
X_pca = pca.fit_transform(X_scaled)

# Apply K-means clustering
kmeans = KMeans(n_clusters=7, random_state=42)
kmeans.fit(X_pca)

# Add cluster labels to the dataset
default['cluster'] = kmeans.labels_ 

# Define key features for overall performance
key_features = ["BICRA",
               "TYPE_Diversified Commercial Banks",
               "TOTAL CAPITAL RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO"]



### 6.5.3. Assign scores from 1 to 10

In [ ]:
# Calculate the performance metric for each cluster
cluster_performance = default.groupby('cluster')[key_features].mean().mean(axis=1) 

# Rank the clusters based on the lowest performance metric will receive a rank of 1 (ascending true)
cluster_rank = cluster_performance.rank(ascending=True).astype(int)

# Assign scores from 1 to 10 based on the cluster rank
default['score'] = default['cluster'].map(cluster_rank)

# Save the updated DataFrame to a new Excel file
default.to_excel('merged_data_with_clusters_and_scores_PCA.xlsx', index=False)

print(default[['cluster', 'score']])

### 6.5.4. Explained Variance vs. Number of Principal Components

In [ ]:
import matplotlib.pyplot as plt

# Fit PCA without specifying n_components
pca = PCA()
pca.fit(X_scaled)

# Calculate the cumulative explained variance ratio
cumulative_explained_variance = np.cumsum(pca.explained_variance_ratio_)

# Plot the cumulative explained variance
plt.figure()
plt.plot(cumulative_explained_variance)
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance vs. Number of Principal Components')

# Add vertical and horizontal lines
ax = plt.gca()
ax.axvline(linestyle='--', color='gray')
ax.axhline(linestyle='--', color='gray')
plt.show()

# You need to get where is flattens for us 12 looks acceptable

## 5.5 Final Model XGBoost Code

In [ ]:
# Add a new column to the DataFrame based scores
default["ROBUST_BANK"] = np.where(default["score"] <= 5, 1, 0)
# DataFrame will have an additional column, ROBUST_BANK, if their score is equal and lower than  with values of either 1 or 0 based on the score column's values.

In [ ]:
### XGBOOST
# pip install xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import plot_importance
import matplotlib.pyplot as plt

# Split the dataset into training and test datasets
train, test = train_test_split(default, test_size=0.3, shuffle=False)

# Define the feature columns
X_cols = ["BICRA",
               "TYPE_Diversified Commercial Banks",
               "TOTAL CAPITAL RATIO",
               "TEXAS RATIO",
               "PROBLEM LOANS RATIO",
               "LOAN LOSS RESERVE TO PROBLEM LOANS",
               "NET INTEREST MARGIN",
               "ROAE",
               "LIQUID ASSETS/ TOTAL DEPOSITS & BORROWINGS ", 
               "TOTAL DEBT/ TOTAL EQUITY",
               "BOOKVALUE PER SHARE GROWTH",
               "OPERATING RETURN ON ASSETS",
               "NON-OPERATING EXPENSE RATIO"]

y = default['ROBUST_BANK']

# Define the feature and target variables for the training and test datasets
X_train = train[X_cols]
y_train = train["ROBUST_BANK"]
X_test = test[X_cols]
y_test = test["ROBUST_BANK"]

# Train the XGBoost model on the training dataset
XGB = XGBClassifier(n_estimators=180, n_jobs=-1, verbose=1)
XGB.fit(X_train, y_train, eval_metric=["auc"], eval_set=[(X_train, y_train), (X_test, y_test)])

# Plot feature importance
plot_importance(XGB)
plt.show()

# Compute and print the area under the receiver operating characteristic (ROC) curve for the training and test datasets
y_train_pred = XGB.predict_proba(X_train)[:, 1]
y_test_pred = XGB.predict_proba(X_test)[:, 1]
print('Training ROC:' + "{:.3f}".format(roc_auc_score(y_train, y_train_pred)))
print('Test ROC:' + "{:.3f}".format(roc_auc_score(y_test, y_test_pred)))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Function to plot the ROC curve
def plot_roc_curve(y_true, y_pred, label):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f'{label} (AUC = {roc_auc:.3f})')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")

# Plot the ROC curve for training and test datasets
plt.figure()
plot_roc_curve(y_train, y_train_pred, 'Training')
plot_roc_curve(y_test, y_test_pred, 'Test')
plt.show()